# Lecture Video Agent — Orchestrator

Runs the pipeline one stage at a time so you can inspect, **edit on disk, and re-load**
your edits before approving and continuing.

**Script Block (implemented):** Planner → Detailed Scripter → Audio-Visual Scene Script.
**Audio branch (implemented):** Audio Generator — Piper TTS + Whisper word-level timestamps, per scene.
**Visual branch (not built yet):** prompt maker → scene code generator → render → critique.

**Editing workflow, script stages:** each stage writes its output to a plain file under
`projects/<slug>/`. Open it in any editor, change whatever you want, save, then run that
stage's *reload* cell — it optionally expands rough edits back to full detail (via
`utils/prompt_enhancer.py`), then injects your edited text back into the pipeline before you
resume to the next stage. Skip the reload cell if you're happy with the output as generated.
The Audio stage has no text reload step — to change a scene's narration, edit
`03_scenes.json`, reload, then re-run Step 4.

In [ ]:
import json
from pathlib import Path

from IPython.display import Audio, HTML, Markdown, Video, display

from state import Scene, VideoState
from workflow import build_graph
from utils.storage import project_dir, save_state
from utils.references import list_reference_files, load_references
from utils.prompt_enhancer import enhance
from agents.planner import revise_plan
from agents.detailed_scripter import revise_script
from agents.scene_script import revise_scenes
from agents.visual_render import RENDERERS

BADGE_COLOR = {
    "manim": "#2f6fed",
    "matplotlib": "#1fa971",
    "image_only": "#c98a1f",
    "audio_only": "#8a5fd1",
}

RENDERED_TYPES = {"manim", "matplotlib"}


def show_scenes(scenes):
    rows = []
    for s in scenes:
        color = BADGE_COLOR.get(s.scene_type, "#888")
        rows.append(f'''
        <div style="border:1px solid #444;border-radius:8px;padding:12px;margin-bottom:10px;">
          <div style="display:flex;justify-content:space-between;align-items:center;">
            <b>Scene {s.id} — {s.subtopic}</b>
            <span style="background:{color};color:white;padding:2px 8px;border-radius:12px;font-size:0.8em;">{s.scene_type}</span>
          </div>
          <p style="margin:8px 0 4px 0;"><i>Animation:</i> {s.animation_description}</p>
          <p style="margin:4px 0 0 0;color:#aaa;"><i>Narration:</i> {s.narration}</p>
        </div>
        ''')
    display(HTML("".join(rows)))


def show_scenes_with_audio(scenes):
    for s in scenes:
        color = BADGE_COLOR.get(s.scene_type, "#888")
        duration = s.timestamps[-1].end if s.timestamps else 0.0
        display(HTML(f'''
        <div style="border:1px solid #444;border-radius:8px;padding:12px;margin-bottom:4px;">
          <div style="display:flex;justify-content:space-between;align-items:center;">
            <b>Scene {s.id} — {s.subtopic}</b>
            <span style="background:{color};color:white;padding:2px 8px;border-radius:12px;font-size:0.8em;">{s.scene_type} · {duration:.1f}s</span>
          </div>
          <p style="margin:8px 0 4px 0;color:#aaa;">{s.narration}</p>
        </div>
        '''))
        if s.audio_path:
            display(Audio(filename=s.audio_path))


def show_scenes_with_video(scenes):
    status_color = {"rendered": "#1fa971", "failed": "#d9534f", "not_applicable": "#666"}
    rendered_scenes = [s for s in scenes if s.scene_type in RENDERED_TYPES]
    other_scenes = [s for s in scenes if s.scene_type not in RENDERED_TYPES]
    for s in rendered_scenes:
        color = status_color.get(s.render_status, "#888")
        renderer_badge = BADGE_COLOR.get(s.scene_type, "#888")
        display(HTML(f'''
        <div style="border:1px solid #444;border-radius:8px;padding:12px;margin-bottom:4px;">
          <div style="display:flex;justify-content:space-between;align-items:center;">
            <b>Scene {s.id} — {s.subtopic}</b>
            <span>
              <span style="background:{renderer_badge};color:white;padding:2px 8px;border-radius:12px;font-size:0.8em;">{s.scene_type}</span>
              <span style="background:{color};color:white;padding:2px 8px;border-radius:12px;font-size:0.8em;">{s.render_status} · {s.render_attempts} attempt(s)</span>
            </span>
          </div>
          <p style="margin:8px 0 4px 0;color:#aaa;">{s.last_feedback or ""}</p>
        </div>
        '''))
        if s.video_path:
            display(Video(filename=s.video_path, embed=True, width=480))
    if other_scenes:
        types = ", ".join(sorted({s.scene_type for s in other_scenes}))
        print(f"{len(other_scenes)} scene(s) ({types}) skipped — no renderer built for them yet.")

## Input

Edit these and re-run the notebook from here.

In [ ]:
topic = "Hopf Oscillators"
audience = "undergraduate engineering students"
duration_minutes = 12
extra_reference_paths: list[str] = []  # any reference files outside the folder below

directory = project_dir(topic)
references_dir = directory / "references"
references_dir.mkdir(exist_ok=True)

initial_state = VideoState(topic=topic, audience=audience, duration_minutes=duration_minutes)

graph = build_graph()
config = {"configurable": {"thread_id": topic}}

print(f"Project folder: {directory}")
print(f"Drop reference files (.txt / .md / .pdf) here any time before Step 2: {references_dir}")

## Optional: supply your own detailed script (skips Steps 1-2 entirely)

If you'd rather write the content yourself than have Claude invent examples and analogies,
point this at a file (markdown or plain text — sections, examples, whatever you want covered)
and run the cell below. It skips Planner and Detailed Scripter completely — no LLM call
generates any content, your file becomes `detailed_script` directly — and jumps straight to
Step 3 (Scene Script), which turns your content into scene-by-scene narration + animation
shots.

**If you use this, skip Steps 1 and 2 below entirely and go straight to Step 3.**

In [ ]:
own_detailed_script_path = ""  # e.g. "my_content.md" — set this, run this cell, then skip to Step 3

if own_detailed_script_path:
    detailed_script_text = Path(own_detailed_script_path).read_text()
    graph.update_state(
        config,
        {
            "topic": topic,
            "audience": audience,
            "duration_minutes": duration_minutes,
            "detailed_script": detailed_script_text,
        },
        as_node="detailed_scripter",
    )
    state = VideoState.model_validate(graph.get_state(config).values)
    print(f"Loaded {len(detailed_script_text)} chars from {own_detailed_script_path}")
    print("Steps 1 and 2 skipped — no LLM call made. Jump straight to Step 3 below.")
    display(Markdown(f"### Your Detailed Script\n\n{state.detailed_script}"))

## Step 1 — Planner *(skip if you used the optional cell above)*

Calls Claude to produce learning objectives and a teaching plan. The graph pauses here.

In [ ]:
result = graph.invoke(initial_state, config=config)
state = VideoState.model_validate(result)

display(Markdown("### Objectives\n" + "\n".join(f"- {o}" for o in state.objectives)))
display(Markdown(f"### Teaching Plan\n\n{state.teaching_plan}"))

In [ ]:
# Optional: instead of hand-editing files, give a free-text instruction and regenerate.
# Re-run this cell as many times as you want (edit `feedback` each time) before moving on.
feedback = ""  # e.g. "focus this on vibration analysis for mechanical engineers"

if feedback:
    revised = revise_plan(state, feedback)
    graph.update_state(config, revised)
    state = VideoState.model_validate(graph.get_state(config).values)
    display(Markdown("### Objectives\n" + "\n".join(f"- {o}" for o in state.objectives)))
    display(Markdown(f"### Teaching Plan\n\n{state.teaching_plan}"))

In [ ]:
# Export — edit these files in any editor if you want changes
objectives_path = directory / "01_objectives.txt"
plan_path = directory / "01_teaching_plan.md"
objectives_path.write_text("\n".join(state.objectives))
plan_path.write_text(state.teaching_plan)
print(f"Wrote {objectives_path}")
print(f"Wrote {plan_path}")

In [ ]:
# Reload — run this only if you edited the file(s) above; otherwise skip it
enhance_edits = True  # expands rough notes back to full detail; set False to use edits verbatim

edited_objectives = [line for line in objectives_path.read_text().splitlines() if line.strip()]
edited_plan = plan_path.read_text()
if enhance_edits:
    edited_plan = enhance(
        edited_plan,
        context="a structured teaching plan for a STEM educational video, sectioned with timing",
    )
graph.update_state(config, {"objectives": edited_objectives, "teaching_plan": edited_plan})
print("Loaded edits back into the pipeline.")

## Human approval gate 1 — Planner

Review (and optionally edit + reload) the plan above. Run the next cell when you approve it —
that resumes the graph into the Detailed Scripter.

## Step 2 — Detailed Scripter *(skip if you used the optional cell above)*

Calls Claude to turn the approved plan (plus any references) into the full spoken content. The graph pauses again after this.

In [ ]:
reference_files = list_reference_files(references_dir) + [
    Path(p) for p in extra_reference_paths
]
references = load_references(reference_files)
graph.update_state(config, {"references": references})
print(f"Loaded {len(references)} reference document(s): {[f.name for f in reference_files]}")

result = graph.invoke(None, config=config)  # resumes from the checkpoint, runs detailed_scripter
state = VideoState.model_validate(result)

display(Markdown(f"### Detailed Script\n\n{state.detailed_script}"))

In [ ]:
# Optional: instead of hand-editing the file, give a free-text instruction and regenerate.
# Re-run this cell as many times as you want (edit `feedback` each time) before moving on.
feedback = ""  # e.g. "add a worked example to the section on eigenvalues"

if feedback:
    revised = revise_script(state, feedback)
    graph.update_state(config, revised)
    state = VideoState.model_validate(graph.get_state(config).values)
    display(Markdown(f"### Detailed Script\n\n{state.detailed_script}"))

In [ ]:
# Export — edit this file if you want changes
script_path = directory / "02_detailed_script.md"
script_path.write_text(state.detailed_script)
print(f"Wrote {script_path}")

In [ ]:
# Reload — run this only if you edited the file above; otherwise skip it
enhance_edits = True  # expands rough notes back to full detail; set False to use edits verbatim

edited_script = script_path.read_text()
if enhance_edits:
    edited_script = enhance(
        edited_script,
        context=(
            "a detailed spoken STEM educational script, organized by subtopic, with "
            "narration, equations, analogies, and examples"
        ),
    )
graph.update_state(config, {"detailed_script": edited_script})
print("Loaded edits back into the pipeline.")

## Human approval gate 2 — Detailed Scripter

Review (and optionally edit + reload) the script above. Run the next cell when you approve it —
that resumes the graph into the Audio-Visual Scene Script agent.

## Step 3 — Audio-Visual Scene Script

Calls Claude to break the script into scenes, each with narration, an animation description, and a manim/react/image_only/audio_only classification.

In [ ]:
result = graph.invoke(None, config=config)  # resumes from the checkpoint, runs scene_script
state = VideoState.model_validate(result)

show_scenes(state.scenes)

In [ ]:
# Optional: instead of hand-editing the JSON, give a free-text instruction and regenerate.
# Re-run this cell as many times as you want (edit `feedback` each time) before moving on.
feedback = ""  # e.g. "split the equation-derivation scene into two shorter scenes"

if feedback:
    revised = revise_scenes(state, feedback)
    graph.update_state(config, revised)
    state = VideoState.model_validate(graph.get_state(config).values)
    show_scenes(state.scenes)

In [ ]:
# Export — edit this file if you want changes (add/remove/reorder scenes, change scene_type, etc.)
scenes_path = directory / "03_scenes.json"
scenes_path.write_text(json.dumps([s.model_dump() for s in state.scenes], indent=2))
print(f"Wrote {scenes_path}")

In [ ]:
# Reload — run this only if you edited the file above; otherwise skip it
enhance_edits = True  # expands rough notes back to full detail; set False to use edits verbatim

raw_scenes = scenes_path.read_text()
if enhance_edits:
    raw_scenes = enhance(
        raw_scenes,
        context=(
            "a JSON array of scenes for a STEM educational video: each has subtopic, "
            "narration (spoken text), animation_description, and scene_type "
            "(manim/react/image_only/audio_only)"
        ),
    )
edited_scenes = [Scene(**item) for item in json.loads(raw_scenes)]
graph.update_state(config, {"scenes": edited_scenes})
state = VideoState.model_validate(graph.get_state(config).values)
show_scenes(state.scenes)

## Human approval gate 3 — Scene Script

Review (and optionally edit + reload) the scenes above. Run the next cell when you approve
them — that resumes the graph into the Audio Generator. Not happy with a scene's narration?
Edit it in `03_scenes.json` above, reload, then run Step 4.

## Step 4 — Audio Generator

Synthesizes narration audio per scene (Piper TTS) and aligns word-level timestamps (Whisper,
local, CPU). No LLM call here — deterministic tool execution. Takes a minute or two for ~20
scenes on CPU.

In [ ]:
result = graph.invoke(None, config=config)  # resumes from the checkpoint, runs audio_generator
state = VideoState.model_validate(result)

show_scenes_with_audio(state.scenes)

## Human approval gate 4 — Audio

Listen to each scene above. If the pacing or pronunciation is off, edit the narration in
`03_scenes.json`, reload it (Step 3's reload cell), then re-run Step 4.

## Step 5 — Visual Render (interactive, one scene at a time)

For each manim/matplotlib scene: renders once (no automated aesthetic critique — that was
burning API calls on a judgment you're about to make yourself). You'll see the clip, then a
text prompt: type feedback to regenerate that scene (as many rounds as you want), or press
Enter to accept and move to the next scene.

**Must be run interactively in Jupyter** (needs your input at each scene) — this cell will hang
if executed headlessly (e.g. `nbconvert --execute`). image_only/audio_only scenes are skipped
automatically (no renderer built for them yet).

In [ ]:
project_root = project_dir(topic)
reviewed_scenes = []

for scene in state.scenes:
    render = RENDERERS.get(scene.scene_type)
    if render is None:
        print(f"Scene {scene.id} ({scene.scene_type}): no renderer yet, skipping.")
        reviewed_scenes.append(scene)
        continue

    scene = render(scene, project_root)
    while True:
        show_scenes_with_video([scene])
        feedback = input(f"Scene {scene.id} — feedback to revise, or Enter to accept: ").strip()
        if not feedback:
            break
        scene = render(scene, project_root, instruction=feedback)
    reviewed_scenes.append(scene)

# Tell the graph this stage is done (without re-running the batch node) and move state forward
graph.update_state(config, {"scenes": reviewed_scenes}, as_node="visual_render")
state = VideoState.model_validate(graph.get_state(config).values)
print(f"\nAll {len(reviewed_scenes)} scenes reviewed.")

## Human approval gate 5 — Rendered clips

Review each clip above, **before audio gets joined to it**. Scenes marked "needs_review"
exhausted their auto-critique retries — look at those first. There's no re-render-single-scene
control yet; for now, fixing a scene means editing its `animation_description` in
`03_scenes.json`, reloading (Step 3's reload cell), and re-running Step 5 for everyone.

## Step 6 — Compose

Merges each scene's video with its narration audio, length-locked to the audio (the source of
truth for timing). Scenes without a rendered clip (image_only/audio_only) get a plain title-card
placeholder for now — there's no illustration generator yet, this just keeps the pipeline
complete end to end. Deterministic — no LLM call, so this is fast and cheap to re-run.

In [ ]:
result = graph.invoke(None, config=config)  # resumes from the checkpoint, runs compose
state = VideoState.model_validate(result)

for s in state.scenes:
    print(f"scene {s.id}: {s.composed_path}")

## Human approval gate 6 — Composed scenes

Spot-check a few composed clips in `projects/<slug>/composed/` if you want (they're plain files,
open them in any player). Run the next cell when ready to stitch everything into the final video.

## Step 7 — Finalize

Stitches every composed scene into the final video and generates subtitles — from the same
word-alignment timestamps that drove animation timing, not a second speech-recognition pass, so
audio/video/subtitles can't drift out of sync with each other.

In [ ]:
result = graph.invoke(None, config=config)  # resumes from the checkpoint, runs finalize
state = VideoState.model_validate(result)

print(f"Final video: {state.final_video_path}")
print(f"Subtitles:   {state.subtitles_path}")
display(Video(filename=state.final_video_path, embed=True, width=640))

## Save

Persists the full pipeline state — topic, plan, script, scenes, all paths — for next time.

In [ ]:
out_path = save_state(state, directory)
print(f"Saved to {out_path}")

## Human approval gate 7 — Final video

Watch the video above. **Not happy with something? Go back and reiterate, don't start over:**
- Wrong content/pacing → edit `01_teaching_plan.md` or `02_detailed_script.md`, reload at that gate, re-run every step after it.
- Wrong scene breakdown or scene type → edit `03_scenes.json`, reload at gate 3, re-run Steps 4-7.
- A specific clip looks wrong → edit that scene's `animation_description` in `03_scenes.json`, reload, re-run Step 5 onward.
- Narration wording/pacing off → edit the scene's `narration` in `03_scenes.json`, reload, re-run Step 4 onward (audio depends on narration text).

Every stage is a plain file — there's no black box step you can't inspect or hand-edit.

Not built yet: real illustrations for image_only/audio_only scenes (currently a plain title card), and a JS/Motion-Canvas renderer for scenes Manim/Matplotlib can't cover.